# Stage 3 — Find the duplicate slides

**The question:** which slides are near-copies of each other, and would
therefore leak between folds?

The dataset contains the same tissue scanned twice, and consecutive sections of
the same biopsy. If one lands in training and its twin in validation, the model
has effectively already seen the answer and every score we record comes out
better than the truth. Stage 4 keeps each group inside a single fold, which
only works if we know the groups.

Output: `data/derived/duplicate_groups.parquet`.

In [1]:
"""Imports.

glob         finds files by pattern; used on Kaggle to locate the outputs of
             earlier notebooks, which arrive as attached datasets.
os           builds file paths and checks whether a file exists on disk.
imagehash    computes the perceptual hash, the "looks like" fingerprint of
             each slide.
numpy        holds the table of distances between every pair of slides.
openslide    opens the whole-slide .tiff files.
pandas       loads the Stage 1 inventory and saves the duplicate groups.
tqdm         draws progress bars for the two long loops.
scipy.sparse coo_matrix stores only the matching pairs instead of a full
             table, and connected_components chains those pairs into groups.
"""

import glob
import os

import imagehash
import numpy as np
import openslide
import pandas as pd
import tqdm
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

In [2]:
"""Paths and settings.

Where things are read from
--------------------------
1. ON_KAGGLE is True inside a Kaggle notebook, where /kaggle/input exists.
2. DATA_DIR holds the competition data. Kaggle mounts it read-only under
   /kaggle/input; locally it is ../data.
3. IMAGE_DIR holds the slide images, one {slide_id}.tiff per slide.
4. find_input(name) locates a file made by an earlier notebook, or by an
   earlier run of this one. It looks in DERIVED_DIR first. On Kaggle every
   notebook runs separately, so earlier output usually arrives as an attached
   dataset instead; in that case it searches every attached dataset, both at
   the top level and inside a derived/ folder. It prints where it found the
   file. If the file is nowhere, it stops with a clear error, unless
   required=False, in which case it returns None.
5. INVENTORY_PATH is the Stage 1 inventory, found with find_input.

Where things are written to
---------------------------
6. DERIVED_DIR is where this project writes its own files. On Kaggle it is
   /kaggle/working/derived/, because /kaggle/input is read-only.
7. CACHE_DIR holds rebuildable bulk data, here the distance table.
8. DUPLICATE_GROUPS_PATH is the file this notebook produces. Notebook 4 reads
   it, so its name and columns must not change.

The distance cache
------------------
9. HASH_SIZE is the fingerprint size: 16 gives 16 x 16 = 256 yes/no answers
   per slide.
10. HAMMING_CACHE_NAME is the cached distance table's name inside the derived
    folder. The hash settings are in the name on purpose: change HASH_SIZE or
    the pyramid level that is read, and the old file no longer matches the
    name, so it is rebuilt instead of silently reused.
11. HAMMING_DISTANCE_PATH is where a freshly computed table is saved.
"""

ON_KAGGLE = os.path.exists("/kaggle/input")

if ON_KAGGLE:
    DATA_DIR = "/kaggle/input/prostate-cancer-grade-assessment/"
    DERIVED_DIR = "/kaggle/working/derived/"
else:
    DATA_DIR = "../data/"
    DERIVED_DIR = os.path.join(DATA_DIR, "derived/")

IMAGE_DIR = os.path.join(DATA_DIR, "train_images/")


def find_input(name, required=True):
    """Return the path of a file made earlier, or None if optional and missing."""
    candidates = [os.path.join(DERIVED_DIR, name)]
    if ON_KAGGLE:
        candidates += sorted(glob.glob(os.path.join("/kaggle/input", "*", name)))
        candidates += sorted(glob.glob(os.path.join("/kaggle/input", "*", "derived", name)))
    for path in candidates:
        if os.path.exists(path):
            print(f"{name}: {path}")
            return path
    if required:
        raise FileNotFoundError(f"{name} not found; run the notebook that makes it, "
                                f"or attach its output as a dataset on Kaggle")
    return None


INVENTORY_PATH = find_input("slide_inventory.parquet")

CACHE_DIR = os.path.join(DERIVED_DIR, "cache/")
DUPLICATE_GROUPS_PATH = os.path.join(DERIVED_DIR, "duplicate_groups.parquet")

HASH_SIZE = 16
HAMMING_CACHE_NAME = f"cache/hamming_phash{HASH_SIZE}_smallest_level.npy"
HAMMING_DISTANCE_PATH = os.path.join(DERIVED_DIR, HAMMING_CACHE_NAME)

slide_inventory.parquet: ../data/derived/slide_inventory.parquet


In [3]:
"""Load the Stage 1 inventory, one row per slide, and show the first rows as a
check. Its row order is the order used for every table in this notebook."""

inventory = pd.read_parquet(INVENTORY_PATH)
inventory.head()

,slide_id,gleason_score,isup_grade,provider,slide_exists,mask_exists,feature_file_exists,feature_tile_count,tiff_pym_lvls,tiff_lvl0_dim,tiff_lvl_ds,tiff_mpx,percentage_empty
0,0005f7aaab2800f6170c399693a96917,0+0,0,karolinska,True,True,None,None,3,"[27648, 29440]","[1.0, 4.0, 16.0]","[0.45201826153776614, 0.45201826153776614]",96.895192
1,000920ad0b612851f8e01bcc880d9b3d,0+0,0,karolinska,True,True,None,None,3,"[15360, 13312]","[1.0, 4.0, 16.0]","[0.45201826153776614, 0.45201826153776614]",94.424705
2,0018ae58b01bdadc8e347995b69f99aa,4+4,4,radboud,True,True,None,None,3,"[5888, 25344]","[1.0, 4.0, 16.0]","[0.4861876369654638, 0.4861876369654638]",82.860363
3,001c62abd11fa4b57bf7a6c603a11bb9,4+4,4,karolinska,True,True,None,None,3,"[23904, 28664]","[1.0, 4.0, 16.00223338916806]","[0.5031982437947761, 0.5031982437947761]",95.209238
4,001d865e65ef5d2579c190a0e0350d8f,0+0,0,karolinska,True,True,None,None,3,"[28672, 34560]","[1.0, 4.0, 16.0]","[0.45201826153776614, 0.45201826153776614]",94.389209


In [4]:
"""The slide ids as a plain Python list, in inventory order. Row i of the
distance table always refers to slide i of this list."""

image_ids = inventory["slide_id"].to_numpy().tolist()

## Hash every slide, then compare every pair

A perceptual hash is a fingerprint of what a picture *looks like*: shrink it
down, ask a fixed list of yes/no questions about it, and keep the answers.
Similar pictures give similar answers, so a near-copy stays close instead of
changing completely the way a file checksum would.

`hash_size=16` gives 256 bits. 64 bits is not enough here — every slide is a
thin pink streak on white, so a coarse fingerprint makes unrelated slides look
identical.

**Hamming distance** is simply how many of the 256 answers differ.

About eight minutes on the first run, then cached.

In [5]:
"""The function that fingerprints every slide and measures how different every
pair of slides is. It is defined here and run in the next cell."""


def compute_hamming_dists():
    """Return a table of Hamming distances between every pair of slides.

    Fingerprinting, one slide at a time
    1. Open the slide's .tiff.
    2. Read its smallest pyramid level in full; it is the only level small
       enough to read whole quickly.
    3. Close the slide in a finally block, so it is closed even if the read
       fails. A loop over 10,616 slides that leaves files open would use up the
       operating system's limit on open files partway through.
    4. Compute the slide's perceptual hash: imagehash shrinks the picture,
       answers HASH_SIZE x HASH_SIZE yes/no questions about its overall
       pattern, and keeps the answers. Similar pictures give similar answers.

    Comparing, every pair once
    5. Make an empty square table, one row and one column per slide.
    6. For each pair of slides i and j with i < j, store how many of the 256
       answers differ. imagehash defines subtraction between two hashes as
       exactly that count, the Hamming distance.
    7. Only the upper-right half of the table, where i < j, is filled in: the
       distance from i to j is the same as from j to i, so the other half would
       repeat it. The lower half stays zero, which does not mean "identical";
       the grouping cell only ever reads the upper half.
    """
    hash_array = []
    for slide_id in tqdm.tqdm(image_ids):
        slide = openslide.OpenSlide(os.path.join(IMAGE_DIR, f"{slide_id}.tiff"))
        try:
            image = slide.read_region((0, 0), slide.level_count - 1,
                                      slide.level_dimensions[slide.level_count - 1])
        finally:
            slide.close()
        hash_array.append(imagehash.phash(image, hash_size=HASH_SIZE))

    num_images = len(hash_array)
    hamming_distances = np.zeros((num_images, num_images))
    for i in tqdm.tqdm(range(num_images)):
        for j in range(num_images):
            if i < j:
                hamming_distances[i, j] = hash_array[i] - hash_array[j]

    return hamming_distances

In [6]:
"""Load the distance table from the cache, or compute it and save it.

1. Look for a cached table made by an earlier run, in the derived folder or,
   on Kaggle, in an attached dataset. It is optional, so a missing cache is not
   an error.
2. If one is found, load it. This takes seconds.
3. If not, compute it, which takes about eight minutes, and save it straight
   away, so the next run can skip the work even if this one stops partway.
4. Print the table's shape as a check: one row and one column per slide.
"""

cached = find_input(HAMMING_CACHE_NAME, required=False)

if cached is not None:
    hamming_distances = np.load(cached)
else:
    hamming_distances = compute_hamming_dists()
    os.makedirs(CACHE_DIR, exist_ok=True)
    np.save(HAMMING_DISTANCE_PATH, hamming_distances)
    print(f"saved {HAMMING_DISTANCE_PATH}")

print(f"distance table: {hamming_distances.shape}, {hamming_distances.dtype}")

cache/hamming_phash16_smallest_level.npy: ../data/derived/cache/hamming_phash16_smallest_level.npy
distance table: (10616, 10616), float64


## Group the matches

Two slides match if their fingerprints differ in 44 bits or fewer. That number
comes from three pieces of evidence, written up in `FINDINGS.md`: grade
agreement stays above 95% up to 42 and 92% at 44 against a coincidence floor of
19%, the first cross-hospital match (which is proof of an error) appears at 46,
and group sizes stay small until about 48.

Matches chain: if A matches B and B matches C, all three are one group.

In [7]:
"""Turn close pairs into duplicate groups.

Finding the close pairs
1. THRESHOLD is the largest distance still counted as a match. 44 was chosen
   from three pieces of evidence, written up in FINDINGS.md.
2. hamming_distances <= THRESHOLD marks every close entry. np.triu with k=1
   keeps only the part strictly above the diagonal, because the lower half of
   the table is all zeros, and reading those zeros as distances would call
   every pair identical.
3. np.where returns the row and column of every remaining match: rows[k] and
   cols[k] are the two slides in the k-th matching pair.

Chaining the pairs into groups
4. Picture every slide as a dot and every matching pair as a string tied
   between two dots. coo_matrix records where the strings go; it stores only
   the pairs, not a full 10,616 x 10,616 table. Its shape is the full slide
   count so the answer comes back with one entry per slide, in inventory
   order.
5. connected_components follows the strings: if A matches B and B matches C,
   all three get the same group number, even though A and C were never
   compared directly. directed=False means a string works in both
   directions, which matters because only the upper half of the table was
   filled in.
6. It returns the number of groups and one group number per slide. The
   numbers themselves are arbitrary labels; slides sharing a number belong
   together.

Recording the result
7. duplicate_group is stored for every slide, not just the duplicated ones: a
   slide with no match is simply a group of one. Stage 4's fold splitter
   needs one group value per row.
8. is_duplicate is True for slides whose group has more than one member.
9. Print a summary: the number of pairs, groups, and duplicated slides, the
   group sizes, and how the duplicated slides split by hospital.
"""

THRESHOLD = 44

close_pairs = np.triu(hamming_distances <= THRESHOLD, k=1)
rows, cols = np.where(close_pairs)
print(f"{len(rows)} pairs at or below distance {THRESHOLD}")

num_slides = len(inventory)
links = coo_matrix(
    (np.ones(len(rows), dtype=np.uint8), (rows, cols)),
    shape=(num_slides, num_slides),
)
num_groups, group_id = connected_components(links, directed=False)
inventory["duplicate_group"] = group_id

group_sizes = inventory["duplicate_group"].value_counts()
inventory["is_duplicate"] = inventory["duplicate_group"].map(group_sizes) > 1

print(f"{num_groups} groups across {num_slides} slides")
print(f"{inventory['is_duplicate'].sum()} slides share a group with someone else "
      f"({100 * inventory['is_duplicate'].mean():.1f}% of the dataset)")
print("groups by size:", group_sizes[group_sizes > 1].value_counts().sort_index().to_dict())
print("grouped slides by hospital:",
      inventory.loc[inventory["is_duplicate"], "provider"].value_counts().to_dict())

285 pairs at or below distance 44
10334 groups across 10616 slides
555 slides share a group with someone else (5.2% of the dataset)
groups by size: {2: 265, 3: 7, 4: 1}
grouped slides by hospital: {'radboud': 505, 'karolinska': 50}


In [8]:
"""Show the first rows of the inventory with its two new columns,
duplicate_group and is_duplicate, as a check."""

inventory.head()

,slide_id,gleason_score,isup_grade,provider,slide_exists,mask_exists,feature_file_exists,feature_tile_count,tiff_pym_lvls,tiff_lvl0_dim,tiff_lvl_ds,tiff_mpx,percentage_empty,duplicate_group,is_duplicate
0,0005f7aaab2800f6170c399693a96917,0+0,0,karolinska,True,True,None,None,3,"[27648, 29440]","[1.0, 4.0, 16.0]","[0.45201826153776614, 0.45201826153776614]",96.895192,0,False
1,000920ad0b612851f8e01bcc880d9b3d,0+0,0,karolinska,True,True,None,None,3,"[15360, 13312]","[1.0, 4.0, 16.0]","[0.45201826153776614, 0.45201826153776614]",94.424705,1,False
2,0018ae58b01bdadc8e347995b69f99aa,4+4,4,radboud,True,True,None,None,3,"[5888, 25344]","[1.0, 4.0, 16.0]","[0.4861876369654638, 0.4861876369654638]",82.860363,2,False
3,001c62abd11fa4b57bf7a6c603a11bb9,4+4,4,karolinska,True,True,None,None,3,"[23904, 28664]","[1.0, 4.0, 16.00223338916806]","[0.5031982437947761, 0.5031982437947761]",95.209238,3,False
4,001d865e65ef5d2579c190a0e0350d8f,0+0,0,karolinska,True,True,None,None,3,"[28672, 34560]","[1.0, 4.0, 16.0]","[0.45201826153776614, 0.45201826153776614]",94.389209,4,False


## Save

One row per slide, all 10,616. Slides with no duplicate get a group of their
own, so Stage 4 can hand `duplicate_group` straight to the fold splitter.

The distance matrix is cached separately. Its filename carries the hash
settings on purpose — change them and the name no longer matches, so it
rebuilds instead of silently reusing a stale file.

In [9]:
"""Save the duplicate groups.

1. Keep three columns: slide_id to join on, duplicate_group for the fold
   splitter, and is_duplicate for reporting. This is Stage 3's own file,
   joined to the inventory by slide_id rather than replacing it.
2. Create the output folder if it does not exist yet. On a fresh Kaggle
   session it will not.
3. Write it as parquet, without pandas' row numbers, which carry no
   information here.
"""

duplicate_groups = inventory[["slide_id", "duplicate_group", "is_duplicate"]]
os.makedirs(DERIVED_DIR, exist_ok=True)
duplicate_groups.to_parquet(DUPLICATE_GROUPS_PATH, index=False)
print(f"wrote {len(duplicate_groups)} rows to {DUPLICATE_GROUPS_PATH}")

wrote 10616 rows to ../data/derived/duplicate_groups.parquet
